<a href="https://colab.research.google.com/github/leman-cap13/NLP_projects/blob/main/NaturalLanguageProgramming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Sentiment Analysis Using Hugging Face Libraries

In [1]:
from datasets import load_dataset


In [2]:
imdb_dataset = load_dataset("stanfordnlp/imdb")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [3]:
imdb_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [4]:
split = imdb_dataset["train"].train_test_split(train_size=0.8, seed=42)

imdb_train_set = split["train"]
imdb_valid_set = split["test"]
imdb_test_set = imdb_dataset["test"]

In [5]:
print(imdb_train_set[1])

{'text': "'The Rookie' was a wonderful movie about the second chances life holds for us and also puts an emotional thought over the audience, making them realize that your dreams can come true. If you loved 'Remember the Titans', 'The Rookie' is the movie for you!! It's the feel good movie of the year and it is the perfect movie for all ages. 'The Rookie' hits a major home run!", 'label': 1}


In [6]:
print(imdb_train_set[1]["text"])


'The Rookie' was a wonderful movie about the second chances life holds for us and also puts an emotional thought over the audience, making them realize that your dreams can come true. If you loved 'Remember the Titans', 'The Rookie' is the movie for you!! It's the feel good movie of the year and it is the perfect movie for all ages. 'The Rookie' hits a major home run!


In [7]:
print(imdb_train_set[1]["label"])

1


#Byte pair encoding (BPE)

In [8]:
import tokenizers


In [9]:
bpe_model = tokenizers.models.BPE(unk_token="<unk>")
bpe_model

BPE(dropout=None, unk_token="<unk>", continuing_subword_prefix=None, end_of_word_suffix=None, fuse_unk=False, byte_fallback=False, ignore_merges=False, vocab={}, merges=[])

In [10]:
bpe_tokenizer = tokenizers.Tokenizer(bpe_model)
bpe_tokenizer

Tokenizer(version="1.0", truncation=None, padding=None, added_tokens=[], normalizer=None, pre_tokenizer=None, post_processor=None, decoder=None, model=BPE(dropout=None, unk_token="<unk>", continuing_subword_prefix=None, end_of_word_suffix=None, fuse_unk=False, byte_fallback=False, ignore_merges=False, vocab={}, merges=[]))

In [11]:
bpe_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()


In [12]:
special_tokens = ["<pad>", "<unk>"] # pad=0 , unk=1


In [13]:
bpe_trainer = tokenizers.trainers.BpeTrainer(
    vocab_size=1000,
    special_tokens=special_tokens
)

In [14]:
train_reviews = [review["text"].lower() for review in imdb_train_set]

train_reviews[0]

'stage adaptations often have a major fault. they often come out looking like a film camera was simply placed on the stage (such as "night mother"). sidney lumet\'s direction keeps the film alive, which is especially difficult since the picture offered him no real challenge. still, it\'s nice to look at for what it is. the chemistry between michael caine and christopher reeve is quite brilliant. the dynamics of their relationship are surprising. caine is fantastic as always, and reeve gets one of his few chances to really act.<br /><br />i confess that i\'ve never seen ira levin\'s play, but i hear that jay presson allen\'s adaptation is faithful. the script is incredibly convoluted, and keeps you guessing. "deathtrap" is an enormously entertaining film, and is recommended for nearly all fans of stage and screen.<br /><br />7.4 out of 10'

In [15]:
bpe_tokenizer.train_from_iterator(train_reviews, bpe_trainer)

In [16]:
some_review = "what an awesome movie! 😊"

bpe_encoding = bpe_tokenizer.encode(some_review)

In [17]:
print(bpe_encoding.tokens)

['what', 'an', 'aw', 'es', 'ome', 'movie', '!', '<unk>']


In [18]:
print(bpe_encoding.ids)

[303, 139, 373, 149, 240, 211, 4, 1]


In [19]:
print(bpe_encoding.type_ids)

[0, 0, 0, 0, 0, 0, 0, 0]


In [20]:
# w h a t
# 0 1 2 3

In [21]:
print(bpe_encoding.offsets)

[(0, 4), (5, 7), (8, 10), (10, 12), (12, 15), (16, 21), (21, 22), (23, 24)]


In [22]:
print(bpe_encoding.attention_mask) # 0->padding 1 -> real token

[1, 1, 1, 1, 1, 1, 1, 1]


In [23]:
print(bpe_encoding.special_tokens_mask)

[0, 0, 0, 0, 0, 0, 0, 0]


In [24]:
print(bpe_tokenizer.decode(bpe_encoding.ids))

what an aw es ome movie !


In [25]:
bpe_encodings = bpe_tokenizer.encode_batch(train_reviews[:3])

In [26]:
bpe_encodings[0].tokens

['st',
 'age',
 'ad',
 'ap',
 't',
 'ations',
 'of',
 'ten',
 'have',
 'a',
 'ma',
 'j',
 'or',
 'fa',
 'ult',
 '.',
 'they',
 'of',
 'ten',
 'come',
 'out',
 'looking',
 'like',
 'a',
 'film',
 'c',
 'amer',
 'a',
 'was',
 'sim',
 'p',
 'ly',
 'pl',
 'ac',
 'ed',
 'on',
 'the',
 'st',
 'age',
 '(',
 'such',
 'as',
 '"',
 'night',
 'mo',
 'ther',
 '"',
 ').',
 'sid',
 'ney',
 'lu',
 'me',
 't',
 "'",
 's',
 'dire',
 'ction',
 'keep',
 's',
 'the',
 'film',
 'al',
 'ive',
 ',',
 'which',
 'is',
 'especially',
 'dif',
 'f',
 'icul',
 't',
 'since',
 'the',
 'pict',
 'ure',
 'off',
 'er',
 'ed',
 'him',
 'no',
 'real',
 'ch',
 'all',
 'en',
 'ge',
 '.',
 'still',
 ',',
 'it',
 "'",
 's',
 'n',
 'ice',
 'to',
 'look',
 'at',
 'for',
 'what',
 'it',
 'is',
 '.',
 'the',
 'che',
 'mis',
 'try',
 'between',
 'm',
 'ich',
 'a',
 'el',
 'c',
 'ain',
 'e',
 'and',
 'chr',
 'is',
 'top',
 'her',
 'ree',
 've',
 'is',
 'quite',
 'br',
 'illi',
 'ant',
 '.',
 'the',
 'd',
 'y',
 'n',
 'am',
 'ics',

In [27]:
bpe_tokenizer.enable_padding(
    pad_id=0,
    pad_token="<pad>"
)

bpe_tokenizer.enable_truncation(max_length=500)

In [28]:
import torch

bpe_encodings = bpe_tokenizer.encode_batch(train_reviews[:3])

bpe_batch_ids = torch.tensor(
    [encoding.ids for encoding in bpe_encodings]
)

print(bpe_batch_ids)

tensor([[159, 402, 176, 246,  61, 782, 156, 737, 252,  42, 239,  51, 154, 460,
         917,  17, 272, 156, 737, 576, 215, 976, 275,  42, 199,  44, 554,  42,
         192, 585,  57, 160, 259, 170, 157, 143, 138, 159, 402,  11, 589, 152,
           5, 819, 168, 230,   5, 521, 924, 981, 962, 250,  61,  10,  60, 426,
         526, 959,  60, 138, 199, 150, 319,  15, 363, 141, 957, 694,  47, 696,
          61, 875, 138, 960, 337, 414, 140, 157, 385, 174, 433, 161, 221, 145,
         213,  17, 549,  15, 151,  10,  60,  55, 416, 146, 407, 144, 182, 303,
         151, 141,  17, 138, 547, 538, 528, 768,  54, 335,  42, 203,  44, 270,
          46, 153, 876, 141, 919, 233, 522, 172, 141, 719, 162, 807, 279,  17,
         138,  45,  66,  55, 188, 989, 156, 378, 698, 301, 296, 689, 212, 558,
         926, 148,  17,  44, 270,  46, 141,  47, 279, 302, 171, 152, 787,  15,
         153, 522, 172, 766, 205, 156, 234, 677, 161, 139, 513, 146, 370, 251,
         219, 162, 197, 162, 166,  50, 265,  47, 266

In [29]:
attention_mask = torch.tensor(
    [encoding.attention_mask for encoding in bpe_encodings]
)

print(attention_mask)

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [30]:
lengths = attention_mask.sum(dim=-1)
print(lengths)

tensor([281, 114, 285])


#Byte-level BPE / BBPE

In [31]:
bpe_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.ByteLevel()


In [32]:
bpe_tokenizer.decoder = tokenizers.decoders.ByteLevel()


In [33]:
special_tokens = ["<pad>", "<unk>"]

bpe_trainer = tokenizers.trainers.BpeTrainer(
    vocab_size=1000,
    special_tokens=special_tokens
)

In [34]:
bpe_tokenizer.train_from_iterator(
    train_reviews,
    bpe_trainer
)

In [35]:
some_review = "what an awesome movie!"

encoding = bpe_tokenizer.encode(some_review)

In [36]:
print(encoding.tokens) #Ġ -> soz baslangici
print(encoding.ids)
print(bpe_tokenizer.decode(encoding.ids))

['Ġwhat', 'Ġan', 'Ġaw', 'es', 'ome', 'Ġmovie', '#']
[354, 216, 561, 148, 244, 232, 4]
 what an awesome movie#


#WordPiece

In [37]:
wordpiece_model = tokenizers.models.WordPiece(
    unk_token="[UNK]"
)

wordpiece_tokenizer = tokenizers.Tokenizer(
    wordpiece_model
)

In [38]:
wordpiece_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()


In [39]:
wordpiece_trainer = tokenizers.trainers.WordPieceTrainer(
    vocab_size=1000,
    special_tokens=["[PAD]", "[UNK]"]
)

In [40]:
wordpiece_tokenizer.train_from_iterator(
    train_reviews,
    wordpiece_trainer
)

In [41]:
some_review = "what an awesome movie! 😊"

encoding = wordpiece_tokenizer.encode(some_review)

print("TOKENS:")
print(encoding.tokens)

print("\nIDS:")
print(encoding.ids)

print("\nDECODED:")
print(wordpiece_tokenizer.decode(encoding.ids))

TOKENS:
['what', 'an', 'aw', '##es', '##ome', 'movie', '!', '[UNK]']

IDS:
[443, 312, 635, 257, 354, 331, 4, 1]

DECODED:
what an aw ##es ##ome movie !


In [42]:
wordpiece_tokenizer.decoder = tokenizers.decoders.WordPiece(
    prefix="##"
)

wordpiece_trainer = tokenizers.trainers.WordPieceTrainer(
    vocab_size=1000,
    special_tokens=["[PAD]", "[UNK]"],
    continuing_subword_prefix="##"
)

wordpiece_tokenizer.train_from_iterator(
    train_reviews,
    wordpiece_trainer
)


some_review = "what an awesome movie! 😊"

encoding = wordpiece_tokenizer.encode(some_review)

print("TOKENS:")
print(encoding.tokens)

print("\nIDS:")
print(encoding.ids)

print("\nDECODED:")
print(wordpiece_tokenizer.decode(encoding.ids))

TOKENS:
['what', 'an', 'aw', '##es', '##ome', 'movie', '!', '[UNK]']

IDS:
[443, 312, 635, 257, 354, 331, 4, 1]

DECODED:
what an awesome movie!


#Unigram LM

In [43]:
unigram_model = tokenizers.models.Unigram()

unigram_tokenizer = tokenizers.Tokenizer(
    unigram_model
)


In [44]:
unigram_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Metaspace(
    replacement="▁",
    prepend_scheme="always"
)


In [45]:
unigram_tokenizer.decoder = tokenizers.decoders.Metaspace(
    replacement="▁",
    prepend_scheme="always"
)

In [46]:
unigram_trainer = tokenizers.trainers.UnigramTrainer(
    vocab_size=1000,
    special_tokens=["<pad>", "<unk>"],
    unk_token="<unk>"
)


In [47]:
unigram_tokenizer.train_from_iterator(
    train_reviews,
    unigram_trainer
)

In [48]:
some_review = "what an awesome movie!😍"

encoding = unigram_tokenizer.encode(some_review)

print("TOKENS:")
print(encoding.tokens)

print("\nIDS:")
print(encoding.ids)

print("\nDECODED:")
print(unigram_tokenizer.decode(encoding.ids, skip_special_tokens=False))

TOKENS:
['▁what', '▁an', '▁a', 'w', 'e', 'some', '▁movie', '!', '😍']

IDS:
[129, 76, 7, 40, 10, 713, 51, 103, 1]

DECODED:
what an awesome movie!<unk>


#Reusing Pretrained Tokenizers

In [49]:
import transformers

In [50]:
gpt2_tokenizer=transformers.AutoTokenizer.from_pretrained("gpt2")
gpt2_tokenizer

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

GPT2Tokenizer(name_or_path='gpt2', vocab_size=50257, model_max_length=1024, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}
)

In [51]:
gpt2_encoding = gpt2_tokenizer(
    train_reviews[:3],
    truncation=True,
    max_length = 500)

In [52]:
gpt2_encoding

{'input_ids': [[14247, 35030, 1690, 423, 257, 1688, 8046, 13, 484, 1690, 1282, 503, 2045, 588, 257, 2646, 4676, 373, 2391, 4624, 319, 262, 3800, 357, 10508, 355, 366, 3847, 2802, 11074, 9785, 1681, 46390, 316, 338, 4571, 7622, 262, 2646, 6776, 11, 543, 318, 2592, 2408, 1201, 262, 4286, 4438, 683, 645, 1103, 4427, 13, 991, 11, 340, 338, 3621, 284, 804, 379, 329, 644, 340, 318, 13, 262, 16585, 1022, 285, 40302, 269, 5718, 290, 33826, 8803, 302, 44655, 318, 2407, 10457, 13, 262, 17262, 286, 511, 2776, 389, 6452, 13, 269, 5718, 318, 9623, 355, 1464, 11, 290, 302, 44655, 3011, 530, 286, 465, 1178, 8395, 284, 1107, 719, 29847, 1671, 1220, 6927, 1671, 11037, 72, 22127, 326, 1312, 1053, 1239, 1775, 4173, 64, 443, 7114, 338, 711, 11, 475, 1312, 3285, 326, 474, 323, 1803, 261, 477, 268, 338, 16711, 318, 17074, 13, 262, 4226, 318, 8131, 47370, 11, 290, 7622, 345, 25260, 13, 366, 22595, 46670, 1, 318, 281, 36005, 17774, 2646, 11, 290, 318, 7151, 329, 3016, 477, 3296, 286, 3800, 290, 3159, 29847, 1

In [53]:
train_reviews[0]

'stage adaptations often have a major fault. they often come out looking like a film camera was simply placed on the stage (such as "night mother"). sidney lumet\'s direction keeps the film alive, which is especially difficult since the picture offered him no real challenge. still, it\'s nice to look at for what it is. the chemistry between michael caine and christopher reeve is quite brilliant. the dynamics of their relationship are surprising. caine is fantastic as always, and reeve gets one of his few chances to really act.<br /><br />i confess that i\'ve never seen ira levin\'s play, but i hear that jay presson allen\'s adaptation is faithful. the script is incredibly convoluted, and keeps you guessing. "deathtrap" is an enormously entertaining film, and is recommended for nearly all fans of stage and screen.<br /><br />7.4 out of 10'

In [54]:
gpt2_token_ids = gpt2_encoding["input_ids"][0][:10]
print(gpt2_token_ids)

[14247, 35030, 1690, 423, 257, 1688, 8046, 13, 484, 1690]


In [55]:
gpt2_tokenizer.decode(gpt2_token_ids)

'stage adaptations often have a major fault. they often'

In [105]:
bert_tokenizer = transformers.AutoTokenizer.from_pretrained("bert-base-uncased")
bert_tokenizer  # Wordpiece tokenizer

BertTokenizer(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [109]:
bert_encodings=bert_tokenizer(
    train_reviews[:3],
    truncation=True,
    padding=True,
    max_length=500,
    return_tensors="pt",
    add_special_tokens=False
)

#101=>CLS
#102 =>SEP


In [110]:
bert_encodings

{'input_ids': tensor([[ 2754, 17241,  2411,  2031,  1037,  2350,  6346,  1012,  2027,  2411,
          2272,  2041,  2559,  2066,  1037,  2143,  4950,  2001,  3432,  2872,
          2006,  1996,  2754,  1006,  2107,  2004,  1000,  2305,  2388,  1000,
          1007,  1012, 11430, 11320, 11368,  1005,  1055,  3257,  7906,  1996,
          2143,  4142,  1010,  2029,  2003,  2926,  3697,  2144,  1996,  3861,
          3253,  2032,  2053,  2613,  4119,  1012,  2145,  1010,  2009,  1005,
          1055,  3835,  2000,  2298,  2012,  2005,  2054,  2009,  2003,  1012,
          1996,  6370,  2090,  2745, 19881,  1998,  5696, 20726,  2003,  3243,
          8235,  1012,  1996, 10949,  1997,  2037,  3276,  2024, 11341,  1012,
         19881,  2003, 10392,  2004,  2467,  1010,  1998, 20726,  4152,  2028,
          1997,  2010,  2261,  9592,  2000,  2428,  2552,  1012,  1026,  7987,
          1013,  1028,  1026,  7987,  1013,  1028,  1045, 18766,  2008,  1045,
          1005,  2310,  2196,  2464, 1

In [106]:
albert_tokenizer = transformers.AutoTokenizer.from_pretrained("albert-base-v2")
albert_tokenizer # unigram

AlbertTokenizer(name_or_path='albert-base-v2', vocab_size=30000, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'bos_token': '[CLS]', 'eos_token': '[SEP]', 'unk_token': '<unk>', 'sep_token': '[SEP]', 'pad_token': '<pad>', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [111]:
albert_encodings = albert_tokenizer(
    train_reviews[:3],
    truncation=True,
    padding=True,
    max_length=500,
    return_tensors="pt"

)


In [112]:
albert_encodings

{'input_ids': tensor([[    2,   876,  5004,    18,   478,    57,    21,   394,  4173,     9,
            59,   478,   340,    70,   699,   101,    21,   171,  3336,    23,
          1659,  1037,    27,    14,   876,    13,     5,  4289,    28,    13,
             7,  4893,   449,     7,     6,     9, 12508,  1612,  5909,    22,
            18,  1400,  8968,    14,   171,  2481,    15,    56,    25,  1118,
          1956,   179,    14,  2151,  1434,    61,    90,   683,  2404,     9,
           174,    15,    32,    22,    18,  2210,    20,   361,    35,    26,
            98,    32,    25,     9,    14,  5427,   128,   832, 22427,    17,
          4479, 24604,    25,  1450,  7472,     9,    14, 12289,    16,    66,
          1429,    50, 12891,     9, 22427,    25, 10356,    28,   550,    15,
            17, 24604,  3049,    53,    16,    33,   310, 11285,    20,   510,
           601,     9,     1,  5145,    13,   118,     1,  5145,    13,   118,
             1,    49, 14586,    30,  

In [113]:
hf_tokenizer=transformers.PreTrainedTokenizerFast(
    tokenizer_object=bpe_tokenizer
)

In [115]:
def collate_fn(batch, tokenizer=bert_tokenizer):  # data => { text : "vfbehfbje", label: 1}
    reviews=[review["text"] for review in batch]
    labels =[review["label"] for review in batch]

    encodings = tokenizer(
        reviews,
        padding=True,
        truncation=True,
        max_length=300,
        return_tensors="pt"
    )

    labels = torch.tensor(labels, dtype=torch.float32).unsqueeze(1) # shape(batch,) => (batch,1)

    return encodings, labels

In [116]:
from torch.utils.data import DataLoader

In [117]:
batch_size=256
imdb_train_loader = DataLoader(
    imdb_train_set,
    batch_size=batch_size,
    collate_fn=collate_fn,
    shuffle=True
)

imdb_valid_loader = DataLoader(
    imdb_valid_set,
    batch_size=batch_size,
    collate_fn = collate_fn,
    shuffle=False

)
imdb_test_loader = DataLoader(
    imdb_test_set,
    batch_size=batch_size,
    collate_fn=collate_fn,
    shuffle=False
)

In [118]:
import torch.nn as nn

In [119]:
class SentimentAnalysisModel(nn.Module):
    def __init__(self, vocab_size, n_layers=2, embed_dim=128, hidden_dim=64, pad_id=0, dropout=0.2):
        super().__init__()

        self.embed = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_id
        )

        self.gru=nn.GRU(  # token, token, token, pad, pad, pad ,pad,........
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout
        )
        self.output =nn.Linear(hidden_dim,1)

    def forward(self, embeddings):
        embeddings = self.embed(embeddings["input_ids"])
        _outputs, hidden_states = self.gru(embeddings)
        return self.output(hidden_states[-1])


In [122]:
import torch
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence


In [123]:
sequences = torch.tensor([
    [1, 2, 0, 0], # 0->pad
    [5, 6, 7, 8]
])

lengths = torch.tensor([2,4])

packed = pack_padded_sequence(
    sequences,
    lengths=lengths,
    enforce_sorted=False,
    batch_first=True
)

print(packed)

PackedSequence(data=tensor([5, 1, 6, 2, 7, 8]), batch_sizes=tensor([2, 2, 1, 1]), sorted_indices=tensor([1, 0]), unsorted_indices=tensor([1, 0]))


In [124]:
padded, lengths = pad_packed_sequence(
    packed,
    batch_first=True
)

print(padded)
print(lengths)

tensor([[1, 2, 0, 0],
        [5, 6, 7, 8]])
tensor([2, 4])


In [125]:
class SentimentAnalysisModel(nn.Module):
    def __init__(self, vocab_size, n_layers=2, embed_dim=128, hidden_dim=64, pad_id=0, dropout=0.2):
        super().__init__()

        self.embed = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_id
        )

        self.gru=nn.GRU(  # token, token, token, pad, pad, pad ,pad,........
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout
        )
        self.output =nn.Linear(hidden_dim,1)

    def forward(self, embeddings):
        input_ids = embeddings["input_ids"]
        attention_mask = embeddings["attention_mask"]

        lengths = attention_mask.sum(dim=1)

        packed_embeddings = pack_padded_sequence(
            self.embed(input_ids),
            lengths=lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _outputs, hidden_states = self.gru(packed_embeddings)
        last_hidden_state = hidden_states[-1]
        logits = self.output(last_hidden_state)

        return logits



#part2

In [56]:
bert_tokenizer = transformers.AutoTokenizer.from_pretrained("bert-base-uncased")
bert_tokenizer

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

BertTokenizer(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [57]:
bert_encoding=bert_tokenizer(train_reviews[:3],
                             padding=True,
                             truncation=True,
                             max_length=500,
                             return_tensors="pt",
                            #  add_special_tokens=False
)

In [58]:
bert_encoding

{'input_ids': tensor([[  101,  2754, 17241,  2411,  2031,  1037,  2350,  6346,  1012,  2027,
          2411,  2272,  2041,  2559,  2066,  1037,  2143,  4950,  2001,  3432,
          2872,  2006,  1996,  2754,  1006,  2107,  2004,  1000,  2305,  2388,
          1000,  1007,  1012, 11430, 11320, 11368,  1005,  1055,  3257,  7906,
          1996,  2143,  4142,  1010,  2029,  2003,  2926,  3697,  2144,  1996,
          3861,  3253,  2032,  2053,  2613,  4119,  1012,  2145,  1010,  2009,
          1005,  1055,  3835,  2000,  2298,  2012,  2005,  2054,  2009,  2003,
          1012,  1996,  6370,  2090,  2745, 19881,  1998,  5696, 20726,  2003,
          3243,  8235,  1012,  1996, 10949,  1997,  2037,  3276,  2024, 11341,
          1012, 19881,  2003, 10392,  2004,  2467,  1010,  1998, 20726,  4152,
          2028,  1997,  2010,  2261,  9592,  2000,  2428,  2552,  1012,  1026,
          7987,  1013,  1028,  1026,  7987,  1013,  1028,  1045, 18766,  2008,
          1045,  1005,  2310,  2196,  

In [59]:
bert_encoding["input_ids"] # starts with 101 [cls]  and ends with 102 [sep]

tensor([[  101,  2754, 17241,  2411,  2031,  1037,  2350,  6346,  1012,  2027,
          2411,  2272,  2041,  2559,  2066,  1037,  2143,  4950,  2001,  3432,
          2872,  2006,  1996,  2754,  1006,  2107,  2004,  1000,  2305,  2388,
          1000,  1007,  1012, 11430, 11320, 11368,  1005,  1055,  3257,  7906,
          1996,  2143,  4142,  1010,  2029,  2003,  2926,  3697,  2144,  1996,
          3861,  3253,  2032,  2053,  2613,  4119,  1012,  2145,  1010,  2009,
          1005,  1055,  3835,  2000,  2298,  2012,  2005,  2054,  2009,  2003,
          1012,  1996,  6370,  2090,  2745, 19881,  1998,  5696, 20726,  2003,
          3243,  8235,  1012,  1996, 10949,  1997,  2037,  3276,  2024, 11341,
          1012, 19881,  2003, 10392,  2004,  2467,  1010,  1998, 20726,  4152,
          2028,  1997,  2010,  2261,  9592,  2000,  2428,  2552,  1012,  1026,
          7987,  1013,  1028,  1026,  7987,  1013,  1028,  1045, 18766,  2008,
          1045,  1005,  2310,  2196,  2464, 11209, 2

In [60]:
bert_encoding["attention_mask"]

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [61]:
albert_tokenizer = transformers.AutoTokenizer.from_pretrained("albert-base-v2")

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/760k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

In [62]:
albert_encoding = albert_tokenizer(train_reviews[:3],
                                  padding=True,
                                  truncation=True,
                                  max_length=500)

In [63]:
albert_encoding

{'input_ids': [[2, 876, 5004, 18, 478, 57, 21, 394, 4173, 9, 59, 478, 340, 70, 699, 101, 21, 171, 3336, 23, 1659, 1037, 27, 14, 876, 13, 5, 4289, 28, 13, 7, 4893, 449, 7, 6, 9, 12508, 1612, 5909, 22, 18, 1400, 8968, 14, 171, 2481, 15, 56, 25, 1118, 1956, 179, 14, 2151, 1434, 61, 90, 683, 2404, 9, 174, 15, 32, 22, 18, 2210, 20, 361, 35, 26, 98, 32, 25, 9, 14, 5427, 128, 832, 22427, 17, 4479, 24604, 25, 1450, 7472, 9, 14, 12289, 16, 66, 1429, 50, 12891, 9, 22427, 25, 10356, 28, 550, 15, 17, 24604, 3049, 53, 16, 33, 310, 11285, 20, 510, 601, 9, 1, 5145, 13, 118, 1, 5145, 13, 118, 1, 49, 14586, 30, 31, 22, 195, 243, 541, 16216, 18863, 22, 18, 418, 15, 47, 31, 990, 30, 3361, 901, 218, 3675, 22, 18, 5004, 25, 10763, 9, 14, 3884, 25, 13003, 1065, 16261, 1427, 15, 17, 8968, 42, 19523, 9, 13, 7, 13921, 16514, 7, 25, 40, 7135, 102, 15677, 171, 15, 17, 25, 5773, 26, 1212, 65, 3047, 16, 876, 17, 2324, 9, 1, 5145, 13, 118, 1, 5145, 13, 118, 1, 465, 9, 300, 70, 16, 332, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [64]:
hf_tokenizer=transformers.PreTrainedTokenizerFast(
    tokenizer_object=bpe_tokenizer
)

In [65]:
hf_encoding=hf_tokenizer(
    train_reviews[:3],
    padding=True,
    truncation=True,
    max_length=500
)

In [66]:
hf_encoding

{'input_ids': [[196, 499, 460, 40, 351, 862, 159, 59, 146, 264, 129, 282, 49, 149, 462, 611, 15, 297, 159, 59, 146, 885, 317, 466, 147, 314, 129, 222, 842, 132, 40, 206, 662, 55, 186, 288, 641, 160, 190, 134, 196, 499, 247, 58, 60, 199, 213, 250, 53, 327, 140, 516, 3, 581, 133, 217, 553, 64, 162, 321, 235, 200, 501, 201, 637, 559, 58, 134, 222, 272, 347, 13, 412, 167, 815, 259, 980, 324, 384, 172, 611, 969, 134, 154, 521, 418, 506, 132, 160, 416, 352, 508, 236, 218, 146, 285, 15, 622, 13, 173, 200, 169, 470, 157, 466, 256, 214, 354, 173, 167, 15, 134, 236, 263, 312, 502, 849, 140, 333, 40, 306, 144, 302, 44, 155, 236, 57, 312, 310, 336, 189, 44, 187, 167, 803, 407, 243, 48, 294, 15, 134, 153, 64, 53, 227, 172, 58, 159, 408, 768, 320, 706, 598, 253, 701, 856, 150, 147, 15, 144, 302, 44, 167, 142, 294, 318, 172, 213, 869, 13, 155, 189, 44, 187, 889, 277, 159, 252, 750, 236, 158, 534, 157, 402, 278, 238, 171, 215, 171, 175, 48, 313, 45, 284, 191, 136, 656, 568, 566, 136, 258, 332, 61, 131

#Building and Training a Sentiment Analysis Model

In [67]:
def collate_fn(batch, tokenizer=bert_tokenizer):
    reviews = [review["text"] for review in batch]
    labels = [review["label"] for review in batch]

    encodings = tokenizer(
        reviews,
        padding=True,
        truncation=True,
        max_length=200,
        return_tensors="pt"
    )

    labels = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)

    return encodings, labels

In [68]:
from torch.utils.data import DataLoader


In [69]:
batch_size=256
imdb_train_loader = DataLoader(
    imdb_train_set,
    batch_size=batch_size,
    collate_fn=collate_fn,
    shuffle=True
)

imdb_valid_loader = DataLoader(
    imdb_valid_set,
    batch_size=batch_size,
    collate_fn = collate_fn,
    shuffle=False

)

imdb_test_loader = DataLoader(
    imdb_test_set,
    batch_size=batch_size,
    collate_fn=collate_fn,
    shuffle=False
)

In [70]:
import torch.nn as nn

In [71]:
class SentimentAnalysisModel(nn.Module):
    def __init__(self, vocab_size, n_layers=2, embed_dim=2, hidden_dim=2, pad_id=0, dropout=0.2):
        super().__init__()

        self.embed= nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_id)

        self.gru = nn.GRU(
            embed_dim,
            hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout

        )
        self.output = nn.Linear(hidden_dim, 1)


    def forward( self, encodings):
        embeddings=self.embed(encodings["input_ids"])
        _outputs,hidden_states = self.gru(embeddings)
        return self.output(hidden_states[-1])


In [72]:
import torch
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

In [73]:
sequences = torch.tensor([
    [1, 2, 0, 0],
    [5, 6, 7, 8]
])

packed = pack_padded_sequence(
    sequences,
    lengths=(2, 4),
    enforce_sorted=False,
    batch_first=True
)

print(packed)

PackedSequence(data=tensor([5, 1, 6, 2, 7, 8]), batch_sizes=tensor([2, 2, 1, 1]), sorted_indices=tensor([1, 0]), unsorted_indices=tensor([1, 0]))


In [74]:
padded, lengths = pad_packed_sequence(
    packed,
    batch_first=True
)

print(padded)
print(lengths)

tensor([[1, 2, 0, 0],
        [5, 6, 7, 8]])
tensor([2, 4])


In [75]:
# import torch
# import torch.nn as nn
# from torch.nn.utils.rnn import pack_padded_sequence


class SentimentAnalysisModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        n_layers=2,
        embed_dim=128,
        hidden_dim=64,
        pad_id=0,
        dropout=0.2
    ):
        super().__init__()

        self.embed = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_id
        )

        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout
        )

        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, encodings):
        input_ids = encodings["input_ids"]
        attention_mask = encodings["attention_mask"]

        embeddings = self.embed(input_ids)

        lengths = attention_mask.sum(dim=1)

        packed_embeddings = pack_padded_sequence(
            embeddings,
            lengths=lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _outputs, hidden_states = self.gru(packed_embeddings)

        last_hidden_state = hidden_states[-1]

        logits = self.output(last_hidden_state)

        return logits

In [126]:
vocab_size = bert_tokenizer.vocab_size
pad_id = bert_tokenizer.pad_token_id

model = SentimentAnalysisModel(
    vocab_size=vocab_size,
    n_layers=2,
    embed_dim=128,
    hidden_dim=64,
    pad_id=pad_id,
    dropout=0.2
)

In [127]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [128]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [129]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for encodings, labels in dataloader:
        encodings = {
            key: value.to(device)
            for key, value in encodings.items()
        }
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(encodings)

        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).float()

        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total

    return avg_loss, accuracy

In [130]:
def evaluate(model, dataloader, criterion, device):
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for encodings, labels in dataloader:
            encodings = {
                key: value.to(device)
                for key, value in encodings.items()
            }
            labels = labels.to(device)

            logits = model(encodings)

            loss = criterion(logits, labels)

            total_loss += loss.item()

            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total

    return avg_loss, accuracy

In [131]:
num_epochs = 5

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(
        model,
        imdb_train_loader,
        optimizer,
        criterion,
        device
    )

    valid_loss, valid_acc = evaluate(
        model,
        imdb_valid_loader,
        criterion,
        device
    )

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Valid Loss: {valid_loss:.4f} | "
        f"Valid Acc: {valid_acc:.4f}"
    )

Epoch 1/5 | Train Loss: 0.6808 | Train Acc: 0.5540 | Valid Loss: 0.6348 | Valid Acc: 0.6350
Epoch 2/5 | Train Loss: 0.6110 | Train Acc: 0.6730 | Valid Loss: 0.6226 | Valid Acc: 0.6514
Epoch 3/5 | Train Loss: 0.5542 | Train Acc: 0.7264 | Valid Loss: 0.5604 | Valid Acc: 0.7132
Epoch 4/5 | Train Loss: 0.4793 | Train Acc: 0.7780 | Valid Loss: 0.4948 | Valid Acc: 0.7758
Epoch 5/5 | Train Loss: 0.4235 | Train Acc: 0.8148 | Valid Loss: 0.6920 | Valid Acc: 0.6890


In [82]:
def predict_sentiment(model, tokenizer, sentences, device, max_length=200):
    model.eval()

    encodings = tokenizer(
        sentences,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )

    encodings = {
        key: value.to(device)
        for key, value in encodings.items()
    }

    with torch.no_grad():
        logits = model(encodings)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).long().squeeze(1)

    label_names = {
        0: "Negative",
        1: "Positive"
    }

    results = []

    for sentence, prob, pred in zip(sentences, probs.squeeze(1), preds):
        results.append({
            "sentence": sentence,
            "probability_positive": prob.item(),
            "predicted_label_id": pred.item(),
            "predicted_label": label_names[pred.item()]
        })

    return results

In [83]:
test_sentences = [
    "This movie was amazing and I really enjoyed it.",
    "The film was boring and too long.",
    "I loved the acting but the story was weak.",
    "Absolutely terrible movie. I would not recommend it.",
    "One of the best films I have watched recently."
]

results = predict_sentiment(
    model=model,
    tokenizer=bert_tokenizer,
    sentences=test_sentences,
    device=device
)

for r in results:
    print("Sentence:", r["sentence"])
    print("Positive probability:", round(r["probability_positive"], 4))
    print("Predicted label id:", r["predicted_label_id"])
    print("Predicted label:", r["predicted_label"])
    print("-" * 50)

Sentence: This movie was amazing and I really enjoyed it.
Positive probability: 0.9466
Predicted label id: 1
Predicted label: Positive
--------------------------------------------------
Sentence: The film was boring and too long.
Positive probability: 0.11
Predicted label id: 0
Predicted label: Negative
--------------------------------------------------
Sentence: I loved the acting but the story was weak.
Positive probability: 0.2772
Predicted label id: 0
Predicted label: Negative
--------------------------------------------------
Sentence: Absolutely terrible movie. I would not recommend it.
Positive probability: 0.104
Predicted label id: 0
Predicted label: Negative
--------------------------------------------------
Sentence: One of the best films I have watched recently.
Positive probability: 0.8275
Predicted label id: 1
Predicted label: Positive
--------------------------------------------------


# Bidirectional RNNs

In [84]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence


class SentimentAnalysisModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        n_layers=2,
        embed_dim=128,
        hidden_dim=64,
        pad_id=0,
        dropout=0.2
    ):
        super().__init__()

        self.embed = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_id
        )

        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout,
            bidirectional=True
        )

        self.output = nn.Linear(2 * hidden_dim, 1)

    def forward(self, encodings):
        input_ids = encodings["input_ids"]
        attention_mask = encodings["attention_mask"]

        embeddings = self.embed(input_ids)

        lengths = attention_mask.sum(dim=1)

        packed_embeddings = pack_padded_sequence(
            embeddings,
            lengths=lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _outputs, hidden_states = self.gru(packed_embeddings)

        n_dims = self.output.in_features

        top_states = hidden_states[-2:].permute(1, 0, 2).reshape(-1, n_dims)

        logits = self.output(top_states)

        return logits

#Reusing Pretrained Embeddings and Language Models

In [85]:
# first part only take first static layer of Bert and use gru model again

In [86]:
bert_model=transformers.AutoModel.from_pretrained("bert-base-uncased")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [87]:
bert_model.embeddings.word_embeddings

Embedding(30522, 768, padding_idx=0)

In [88]:
bert_model.embeddings.word_embeddings.weight.data

tensor([[-0.0102, -0.0615, -0.0265,  ..., -0.0199, -0.0372, -0.0098],
        [-0.0117, -0.0600, -0.0323,  ..., -0.0168, -0.0401, -0.0107],
        [-0.0198, -0.0627, -0.0326,  ..., -0.0165, -0.0420, -0.0032],
        ...,
        [-0.0218, -0.0556, -0.0135,  ..., -0.0043, -0.0151, -0.0249],
        [-0.0462, -0.0565, -0.0019,  ...,  0.0157, -0.0139, -0.0095],
        [ 0.0015, -0.0821, -0.0160,  ..., -0.0081, -0.0475,  0.0753]])

In [89]:
nn.Embedding.from_pretrained(
    bert_model.embeddings.word_embeddings.weight.data,
    freeze=True
)

Embedding(30522, 768)

In [90]:
class SentimentAnalysisModelPreEmbeds(nn.Module):
    def __init__(self, pretrained_embeddings, n_layers=2, hidden_dim=64, dropout=0.2 ):
        super().__init__()

        weights = pretrained_embeddings.weight.data

        self.embed = nn.Embedding.from_pretrained(
            weights,
            freeze=True
        )

        embed_dim = weights.shape[-1]

        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout
        )

        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, encodings):
        embeddings = self.embed(encodings["input_ids"])

        lengths = encodings["attention_mask"].sum(dim=1)

        packed = pack_padded_sequence(
            embeddings,
            lengths=lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _outputs, hidden_states = self.gru(packed)

        return self.output(hidden_states[-1])

In [91]:
imdb_model_bert_embeds = SentimentAnalysisModelPreEmbeds(
    bert_model.embeddings.word_embeddings
).to(device)


In [92]:
# use contextual bert model

In [93]:
bert_encoding = bert_tokenizer(
    train_reviews[:3],
    padding=True,
    max_length=200,
    truncation=True,
    return_tensors="pt"
)
# bert_model=transformers.AutoModel.from_pretrained("bert-base-uncased")

bert_output = bert_model(**bert_encoding)

In [94]:
transformers.AutoModel.from_pretrained("bert-base-uncased").config.hidden_size

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


768

In [95]:
class SentimentAnalysisModelBert(nn.Module):
    def __init__(self, n_layers=2, hidden_dim=64, dropout=0.2):
        super().__init__()

        self.bert = transformers.AutoModel.from_pretrained("bert-base-uncased")

        embed_dim = self.bert.config.hidden_size

        self.gru = nn.GRU(
            embed_dim,
            hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout
        )

        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, encodings):
        contextualized_embeddings = self.bert(**encodings).last_hidden_state

        lengths = encodings["attention_mask"].sum(dim=1)

        packed = pack_padded_sequence(
            contextualized_embeddings,
            lengths=lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _outputs, hidden_states = self.gru(packed)

        return self.output(hidden_states[-1])

In [96]:
#pooler layer

In [97]:
class SentimentAnalysisModelBertCLS(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = transformers.AutoModel.from_pretrained("bert-base-uncased")
        self.output = nn.Linear(self.bert.config.hidden_size, 1)

    def forward(self, encodings):
        bert_output = self.bert(**encodings)
        cls_vector = bert_output.pooler_output
        return self.output(cls_vector)

In [98]:
#cls token

In [99]:
def forward(self, encodings):
    bert_output = self.bert(**encodings)
    return self.output(bert_output.last_hidden_state[:, 0])


#Task-Specific Classes

In [100]:
from transformers import BertForSequenceClassification

bert_for_binary_clf = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    dtype=torch.float16
).to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [101]:
import torch
from transformers import BertForSequenceClassification

torch.manual_seed(42)

bert_for_binary_clf = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    dtype=torch.float16
).to(device)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [102]:
review = ["This was a great movie!"]

encoding = bert_tokenizer(
    review,
    padding=True,
    truncation=True,
    max_length=200,
    return_tensors="pt"
)

encoding = {
    key: value.to(device)
    for key, value in encoding.items()
}

with torch.no_grad():
    output = bert_for_binary_clf(
        input_ids=encoding["input_ids"],
        attention_mask=encoding["attention_mask"]
    )

print(output.logits)

probs = torch.softmax(output.logits, dim=-1)

print(probs)

tensor([[-0.0120,  0.6304]], device='cuda:0', dtype=torch.float16)
tensor([[0.3447, 0.6553]], device='cuda:0', dtype=torch.float16)


In [103]:
review = ["This was a great movie!"]

encoding = bert_tokenizer(
    review,
    padding=True,
    truncation=True,
    max_length=200,
    return_tensors="pt"
)

encoding = {
    key: value.to(device)
    for key, value in encoding.items()
}

with torch.no_grad():
    output = bert_for_binary_clf(**encoding)

logits = output.logits
probs = torch.softmax(logits, dim=-1)
predicted_class = torch.argmax(probs, dim=-1).item()

print("Logits:", logits)
print("Probabilities:", probs)
print("Prediction:", "positive" if predicted_class == 1 else "negative")

Logits: tensor([[-0.0120,  0.6304]], device='cuda:0', dtype=torch.float16)
Probabilities: tensor([[0.3447, 0.6553]], device='cuda:0', dtype=torch.float16)
Prediction: positive


In [104]:
with torch.no_grad():
    output = bert_for_binary_clf(
        input_ids=torch.tensor(encoding["input_ids"], device=device),
        attention_mask=torch.tensor(encoding["attention_mask"], device=device),
        labels=torch.tensor([1], device=device)
    )
output.loss

/tmp/ipykernel_7617/3259999752.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_ids=torch.tensor(encoding["input_ids"], device=device),
/tmp/ipykernel_7617/3259999752.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  attention_mask=torch.tensor(encoding["attention_mask"], device=device),


tensor(0.4226, device='cuda:0', dtype=torch.float16)